# ElGamal Proof Repair Experiment

Adapted for the **`shannon-llm-integration`** branch. Differences from the
original notebook are called out below, because several are load-bearing.

## What this branch adds

| Feature | Effect here |
|---|---|
| **`elgamal-changelog-repair` spec** | Replays the corpus's own tactic script and only calls the LLM at the first tactic that no longer applies. Lemmas that still compile cost **zero LLM calls**. |
| **Three providers** | `deepseek`, `anthropic` (Claude), or `lm_studio` (local Gemma etc.) — one switch, same harness. |
| **Spend cap** | `cost_limit_usd` is enforced *during* the run (`integration/agent/budget.py`), not reported after it. |
| **Error-kind routing** | `ec_errors.py` classifies each failure; tactic failures now retrieve *tactic* changelog entries instead of import notes. |
| **Repair metrics** | `summary.json` carries replay fractions, hop distribution, and hint uptake. |
| **Version detection** | Target EasyCrypt is detected via `git describe` (**r2026.06** here), not hardcoded. |

## What is new since (docs/IMPLEMENTATION_PROGRESS.md §11)

None of this needs a flag — it is all on the default path, so this notebook
exercises it as written. Where to look for each is in the last column.

| Change | What it does | Visible in |
|---|---|---|
| **Error-kind rule selection** | Import-repair rules are ordered by relevance to the error EasyCrypt actually reported, and re-classified after every accepted rule, so targeting follows the file rather than its first error. | "Inspect one trial" → `selected_for` / `relevance` |
| **Graded progress (W4.5)** | `improved` (did the first error move later?) replaced by an outcome ladder. `reached_proof` means the load errors are gone and a **tactic** is now at fault — import repair finishing, even though EasyCrypt still exits nonzero. | "Results" → outcome distribution |
| **Minimisation** | The manifest went 15 → 116 rules, so "keep anything that does not hurt" started dragging unrelated theories into the file. Rules that are not needed are now taken back out. | "Inspect one trial" → dropped rules |
| **Symbol moves 1 → 5** | The history miner tracked 16 hand-picked theories out of ~127, so a move was only visible when both ends were tracked. Now 135, with two guards against coincidence. | `proof_corpus/ec_migrations.toml` |
| **14 authored repair notes** (was 4) | Derived notes state facts; authored ones explain *why* a thing broke and what to do instead. Includes the r2024.09 cost-logic removal, which fails as a **parse error** rather than an unknown symbol. | the hint block shown to the model |

**Not exercised here:** version hopping (`--version-hop`). It builds an
EasyCrypt per release — an opam switch and a full OCaml build each — so it
stays on the CLI. See `docs/IMPLEMENTATION_PROGRESS.md` §9.2.

## Two spec choices

- **`elgamal-changelog-repair`** (default) — replay-until-failure + changelog hints. Cheaper and measures version drift directly.
- **`elgamal-broken-repair`** — the original notebook's spec: admit everything, hand the model the broken script, rebuild from scratch. Every trial costs LLM calls.

## Prerequisites

- **Embeddings**: LM Studio running with an embedding model loaded — required for *every* provider (Anthropic has no embeddings API).
  ```bash
  export PATH="$HOME/.lmstudio/bin:$PATH"
  lms load text-embedding-nomic-embed-text-v1.5
  ```
- **Credentials**: `DEEPSEEK_API_KEY` and/or `ANTHROPIC_API_KEY`. `.env` is **not** auto-loaded — the next cell sources it.
- **EasyCrypt**: the patched fork built at `integration/extern/easycrypt/_build/default/src/ec.exe`.

> ⚠️ **Paid providers.** The CLI requires an interactive human confirmation before spending. This notebook is a *human-driven* surface, so it calls `run_experiment` directly — the `cost_limit_usd` cap below is what bounds the spend. Set it deliberately. Per [`AGENTS.md`](../AGENTS.md), an automated agent must never answer the CLI's confirmation prompt on your behalf.

In [1]:
import logging
import os
import sys
from pathlib import Path

# Project root on the path, and cwd there (corpus paths are root-relative).
PROJECT_ROOT = Path(os.getcwd()).parent if Path(os.getcwd()).name == "notebooks" else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

# .env is not auto-loaded anywhere in this repo (no dotenv dependency).
env_path = PROJECT_ROOT / ".env"
if env_path.is_file():
    for raw in env_path.read_text().splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
# The embeddings endpoint is called thousands of times per trial; its request
# log drowns everything else in this notebook.
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("openai").setLevel(logging.WARNING)

print(f"Project root      : {PROJECT_ROOT}")
print(f"DEEPSEEK_API_KEY  : {'set' if os.environ.get('DEEPSEEK_API_KEY') else 'NOT SET'}")
print(f"ANTHROPIC_API_KEY : {'set' if os.environ.get('ANTHROPIC_API_KEY') else 'NOT SET'}")

Project root      : /home/m8simmon/cs846/AI4EC
DEEPSEEK_API_KEY  : set
ANTHROPIC_API_KEY : set


## Preflight

Fails fast on the two things that otherwise break a run *after* it has started spending.

In [2]:
import json
import re
from pathlib import Path

from openai import APIConnectionError, AuthenticationError

from integration.agent.config import (
    THINKING_HIGH_UNLESS_STUCK,
    AgentConfig,
    resolve_effort_for_step,
    resolve_thinking_for_step,
)
from integration.agent.ec_program import parse_program_block
from integration.agent.goal_diff import format_state_diff
from integration.agent.llm import TRANSPORT_ERRORS
from integration.agent.prompt import (
    _seq_position_bullets,
    format_broken_tactic_repair,
)

pair = parse_program_block(
    Path("integration/tests/fixtures/elgamal_equiv_block.txt").read_text())
txt = " ".join(_seq_position_bullets(pair))
lad = format_broken_tactic_repair(
    "seq 4 3 : (inv).", "[critical] invalid `position' parameter")
cfg = AgentConfig(llm_provider="deepseek", llm_thinking=THINKING_HIGH_UNLESS_STUCK)
stuck = [{"outcome": o} for o in
         ("failed", "no_op", "accepted", "undone", "failed", "accepted")]

assert AgentConfig().history_steps == 15, "history_steps not 15"
assert (len(pair.left), len(pair.right)) == (13, 12), "ec_program stale"
assert "can never exceed" in txt and "N must be 0..13" not in txt, "seq wording stale"
assert "^<K" in txt, "missing ^<K guidance"
assert "Re-read the instruction counts" not in lad, "ladder stale"
assert "ASYMMETRIC" in lad, "asymmetric note missing"
assert APIConnectionError in TRANSPORT_ERRORS, "transport fix stale"
assert AuthenticationError not in TRANSPORT_ERRORS, "auth must fail fast"
assert resolve_effort_for_step(cfg, [{"outcome": "accepted"}] * 5) == "high"
assert resolve_effort_for_step(cfg, stuck) is None, "effort not released"
assert resolve_thinking_for_step(cfg, stuck) == "enabled"

# The config cell, found by SPEC_NAME (unique to it) rather than by index --
# inserting this cell shifts every index, and matching on THINKING_MODE alone
# finds this cell's own text.
nb = json.loads(Path("notebooks/elgamal_repair_experiment.ipynb").read_text())
cfg_src = next(
    "".join(c["source"]) for c in nb["cells"]
    if c["cell_type"] == "code"
    and re.search(r"^SPEC_NAME\s*=", "".join(c["source"]), re.M)
)
mode_line = next(l.strip() for l in cfg_src.splitlines()
                 if re.match(r"^THINKING_MODE\s*=", l))
assert re.match(r'^THINKING_MODE\s*=\s*"adaptive"', mode_line), \
    f"config cell still on the old thinking mode: {mode_line}"

print("OK - all changes live")


OK - all changes live


In [3]:
from integration.agent.config import AgentConfig
from integration.agent.ec_version import detect_target_version
from integration.experiment.__main__ import _embeddings_endpoint_status

_probe = AgentConfig()

ok, detail = _embeddings_endpoint_status(_probe)
print(f"Embeddings endpoint : {'OK' if ok else 'UNAVAILABLE'} — {detail}")
if not ok:
    print("  -> start LM Studio and load an embedding model; every provider needs it.")

print(f"EasyCrypt binary    : {'found' if _probe.easycrypt_bin.exists() else 'MISSING'} ({_probe.easycrypt_bin})")
_target = detect_target_version(_probe.easycrypt_bin)
print(f"EasyCrypt version   : {_target.version} (via {_target.method}, confidence {_target.confidence})")

Embeddings endpoint : OK — 1 model(s) available
EasyCrypt binary    : found (/home/m8simmon/cs846/AI4EC/integration/extern/easycrypt/_build/default/src/ec.exe)
EasyCrypt version   : r2026.06 (via git_describe, confidence high)


## Configuration

In [4]:
# --- Spec -------------------------------------------------------------------
SPEC_NAME = "elgamal-changelog-repair"   # or "elgamal-broken-repair"

# --- Provider ---------------------------------------------------------------
PROVIDER = "deepseek"                    # deepseek | anthropic | lm_studio

# deepseek: deepseek-v4-flash (cheap) | deepseek-v4-pro
# anthropic: claude-opus-5 (default) | claude-sonnet-5
# lm_studio: whatever is loaded locally
MODEL = "deepseek-v4-flash"

THINKING_MODE = "adaptive"               # disabled | enabled | adaptive |
                                         # high_unless_stuck
# Keep thinking on. Matched on the same three proofs, same model, adaptive
# accepted 43% of productive calls against 4% for disabled -- and all
# three thinking-disabled trials went STUCK having accepted ~nothing.
#
# `high_unless_stuck` exists (effort `high` while converging, released once
# >=4 of the last 6 steps are unproductive) but was MEASURED AND REVERTED.
# Matched on G2_G3's first 13 steps, same lemma, same model:
#     adaptive           median 121s   max  289s
#     high_unless_stuck  median 147s   max  914s
# It barely moves the typical step (+21%) and TRIPLES the tail, and with a
# 600s client timeout plus the SDK's 2 retries one bad call can occupy 30
# minutes. No evidence it improves outcomes: INDCPA_Security took 4 steps
# against a 2-7 historical range. Throughput is the binding constraint on
# the only two lemmas that matter, so depth is not worth buying here.
REASONING_EFFORT = None                  # deepseek: high|max ; anthropic: low..max
EMBED_MODEL = "text-embedding-nomic-embed-text-v1.5"

# --- Budget -----------------------------------------------------------------
MAX_TRIALS = 15                          # the whole corpus (15 proofs), shortest first
# Step budget = multiplier x tactic lines.
# Raised 1.4 -> 2.5 on run D's evidence: ALL THREE unfinished lemmas exhausted
# their budget exactly (G2_G3 42/42, INDCPA_HEG_G1 77/77, G1_G2_eq 145/145), so
# every one was stopped by the harness rather than by the model. G1_G2_eq is the
# clearest case -- 107 accepted against 24 failed, the healthiest ratio in the
# run, cut off with 51 tactics already added.
# Cost: projected from run D's measured $/step, 2.5x lands at ~$2.65 against the
# unchanged $5.00 cap. Treat that as a FLOOR, not a ceiling -- cost per step
# grows as the trajectory lengthens the prompt. The cap is the real protection
# and it is checked before every call.
ADAPTIVE_MULTIPLIER = 2.5
MIN_STEPS = 10                           # floor for the step budget
STUCK_LIMIT = 20
TOP_K_PREMISES = 10
# Per-request client timeout, shared by the chat and embedding clients.
# Cut 600 -> 180: the OpenAI SDK retries twice, so 600s meant a single
# wedged call could hold a trial for 30 minutes. Since transport errors are
# now retried rather than fatal (llm.py TRANSPORT_ERRORS), failing fast and
# retrying is strictly better than waiting. Embedding batches finish in
# seconds, so this is ample for them.
LLM_TIMEOUT_S = 180
LLM_MAX_TOKENS = 32768                   # mean output was ~13.2k against the
                                         # old 16384 cap (82% of ceiling), so
                                         # calls truncated routinely. Truncation
                                         # is a budget problem, not a reason to
                                         # turn thinking off.
# Enforced DURING the run, checked before each call, so actual spend can
# overshoot by at most one call. None = uncapped.
# 1.00 was too low to finish: the last Claude run stopped early at 12 of 15
# trials, and a truncated run cannot be compared with a complete one.
COST_LIMIT_USD = 5.00

# --- Knowledge base ---------------------------------------------------------
# False turns the changelog/repair_doc hints off. Use it to check whether a
# hint block is CAUSING a failure -- that is a debugging question and one run
# answers it.
#
# It is NOT an A/B you should run from here. An A/B on this corpus was already
# dropped for variance: 11-vs-1 accepted tactics under identical configuration
# (docs/ELGAMAL_E2E_RESULTS.md §6). One notebook run per arm cannot separate
# anything. See docs/IMPLEMENTATION_PROGRESS.md §11 item 6.
CHANGELOG_HINTS = True

DATA_DIR = Path("data")
OUTPUT_DIR = None                        # None = auto-timestamped

In [5]:
from integration.agent.config import (
    LLM_PROVIDER_ANTHROPIC,
    LLM_PROVIDER_DEEPSEEK,
    PAID_LLM_PROVIDERS,
    apply_anthropic_provider,
    apply_deepseek_provider,
)
from integration.experiment.config import ExperimentConfig

agent = AgentConfig(
    top_k=TOP_K_PREMISES,
    llm_max_tokens=LLM_MAX_TOKENS,
    embed_model=EMBED_MODEL,
    lm_studio_timeout=LLM_TIMEOUT_S,
)

if PROVIDER == LLM_PROVIDER_DEEPSEEK:
    apply_deepseek_provider(agent, model=MODEL, thinking=THINKING_MODE,
                            reasoning_effort=REASONING_EFFORT)
elif PROVIDER == LLM_PROVIDER_ANTHROPIC:
    apply_anthropic_provider(agent, model=MODEL, thinking=THINKING_MODE,
                             reasoning_effort=REASONING_EFFORT)
else:
    agent.llm_model = MODEL  # local: free, no cap needed

exp_config = ExperimentConfig(
    spec_name=SPEC_NAME,
    trials=MAX_TRIALS,
    stuck_limit=STUCK_LIMIT,
    data_dir=DATA_DIR,
    agent=agent,
    sort_by_difficulty=True,
    adaptive_steps_multiplier=ADAPTIVE_MULTIPLIER,
    min_adaptive_steps=MIN_STEPS,
    # A cap is meaningless for a local model and is refused for a model with
    # no published rates, so only set it where it can actually be enforced.
    cost_limit_usd=COST_LIMIT_USD if PROVIDER in PAID_LLM_PROVIDERS else None,
)
if OUTPUT_DIR is not None:
    exp_config.output_dir = Path(OUTPUT_DIR)

exp_config = exp_config.with_agent_defaults()   # builds the SpendBudget

print(f"Spec            : {exp_config.spec_name}")
print(f"Provider/model  : {agent.llm_provider} / {agent.llm_model}")
print(f"Thinking/effort : {agent.llm_thinking} / {agent.llm_reasoning_effort}")
print(f"Output dir      : {exp_config.output_dir}")
print(f"Adaptive steps  : {ADAPTIVE_MULTIPLIER}x (min {MIN_STEPS})")
print(f"Spend cap       : {exp_config.agent.spend_budget.status() if exp_config.agent.spend_budget else 'none (uncapped)'}")

Spec            : elgamal-changelog-repair
Provider/model  : deepseek / deepseek-v4-flash
Thinking/effort : adaptive / None
Output dir      : /home/m8simmon/cs846/AI4EC/integration/output/experiments/run-20260807T151318Z
Adaptive steps  : 2.5x (min 10)
Spend cap       : $0.0000 of $5.00 spent (0 calls, $5.0000 remaining)


## Preview: proof cases, shortest first

In [6]:
from integration.experiment.corpora.elgamal import ElGamalCorpus

corpus = ElGamalCorpus(data_dir=DATA_DIR, sandbox_dir=exp_config.output_dir / "sandboxes")
all_cases = sorted(corpus.load_cases(), key=lambda c: len(c.tactic_lines))

print(f"Available proofs: {len(all_cases)}   |   attempting: {min(MAX_TRIALS, len(all_cases))}\n")
print(f"{'#':<3} {'Name':<20} {'Lines':>6} {'Steps':>7}")
print("-" * 40)
for i, case in enumerate(all_cases):
    n = len(case.tactic_lines)
    marker = " <--" if i < MAX_TRIALS else ""
    print(f"{i:<3} {case.name:<20} {n:>6} {exp_config.steps_for_case(n):>7}{marker}")

Available proofs: 15   |   attempting: 15

#   Name                  Lines   Steps
----------------------------------------
0   enc_stateless             1      10 <--
1   INDCPA_Sec                1      10 <--
2   INDCPA_Security           2      10 <--
3   log_gen                   3      10 <--
4   gen_log                   5      12 <--
5   grexpAll                  5      12 <--
6   RO_track_f_ll             8      20 <--
7   G1_G2                     9      22 <--
8   G3_true                  16      40 <--
9   RO_LCDHAdv               18      45 <--
10  G2_bad_ub                19      47 <--
11  G2_G3                    30      75 <--
12  INDCPA_HEG_G1            55     137 <--
13  correctness              58     145 <--
14  G1_G2_eq                104     260 <--


## Confirm paid usage

`run_experiment` does **not** gate paid providers — the confirmation lives in
the CLI entry points only, so calling it directly from a notebook would spend
real money with no prompt. This cell reproduces the CLI's gate.

`sys.stdin.readline()` does not work under ipykernel, so this uses `input()`,
which Jupyter routes to the frontend prompt.


In [7]:
from integration.agent.config import PAID_LLM_PROVIDERS
from integration.experiment.paid_confirm import (
    CONFIRMATION_PHRASE,
    format_paid_provider_warning,
)

CONFIRMED = agent.llm_provider not in PAID_LLM_PROVIDERS
if CONFIRMED:
    print(f"{agent.llm_provider}: not a paid provider, no confirmation needed.")
else:
    print(format_paid_provider_warning(
        config=agent,
        trials=MAX_TRIALS,
        informal=False,
    ))
    CONFIRMED = input().strip() == CONFIRMATION_PHRASE
    print("Confirmed." if CONFIRMED else "NOT confirmed - the run cell will refuse.")



Provider                 : DeepSeek (https://api.deepseek.com)
Chat model               : deepseek-v4-flash
Rates (USD / 1M tokens)  : input $0.1400, cached input $0.0028, output $0.2800
Thinking                 : adaptive (window=5)
Reasoning effort         : (unset; used when adaptive enables thinking)
Max output tokens        : 32768
Trials (iterations)      : 15
Max agent steps / trial  : 200
Upper bound solver calls : 15 × 200 = 3000
SPEND CAP                : $5.00 USD (run stops when reached; may overshoot by one call)

Embeddings still use the local LM Studio endpoint (not DeepSeek):
  http://localhost:1234/v1

Agents / automated tools MUST NEVER answer this prompt for the user. Only a human may confirm paid API usage.

Type YES (all caps) to proceed, or anything else to abort.

Confirmed.


## Run

Shortest-first, stopping as soon as `COST_LIMIT_USD` is reached.

> With `elgamal-changelog-repair`, trials whose proof still replays verbatim
> finish in seconds with **zero** LLM calls — those are free.

In [8]:
from dataclasses import replace as dc_replace

from integration.experiment.__main__ import _build_spec, _with_sandbox_dir
from integration.experiment.runner import run_experiment

exp_config.output_dir.mkdir(parents=True, exist_ok=True)
spec = _with_sandbox_dir(
    _build_spec(exp_config.spec_name, DATA_DIR),
    DATA_DIR,
    exp_config.output_dir / "sandboxes",
)

if spec.replay_bootstrap is not None and not CHANGELOG_HINTS:
    # Off for the whole chain, including the per-failure refresh: a run that
    # refetched hints on the next failure is not a hints-off run.
    spec = dc_replace(
        spec,
        replay_bootstrap=dc_replace(spec.replay_bootstrap, changelog_hints=False),
    )
    print("KNOWLEDGE BASE DISABLED - this is the hints-off arm.\n")

if not CONFIRMED:
    raise RuntimeError(
        "Paid usage was not confirmed - run the confirmation cell above. "
        "No API call was made."
    )

result = run_experiment(spec, exp_config)

print(f"\nSpec      : {result.spec_name}   mode: {result.mode}")
print(f"Trials    : {result.trials_run} run, {result.trials_skipped} skipped")
print(f"Outcomes  : {result.successes} complete, {result.stuck} stuck, {result.max_steps} max-steps, {result.errors} errors")
if result.estimated_cost:
    print(f"Cost      : ${result.estimated_cost['usd']:.6f}")
if result.budget:
    print(f"Budget    : {result.budget['spent_usd']:.6f} / {result.budget['limit_usd']:.2f} USD"
          f"{'  (STOPPED EARLY)' if result.budget_stopped else ''}")
print(f"Output    : {result.output_dir}")

INFO: Trial 0: enc_stateless (1 tactic lines -> 10 step budget)
INFO: Trial 1: INDCPA_Sec (1 tactic lines -> 10 step budget)
INFO: Trial 2: INDCPA_Security (2 tactic lines -> 10 step budget)
INFO: Tactic failed: [critical] [/home/m8simmon/cs846/AI4EC/integration/output/experiments/run-20260807T151318Z/trials/trial_002/agent_work.agent.ec: line 828 (24-37)] expecting a `memory', not a `formula'
INFO: Tactic failed: [critical] [/home/m8simmon/cs846/AI4EC/integration/output/experiments/run-20260807T151318Z/trials/trial_002/agent_work.agent.ec: line 828 (27-40)] too many arguments
INFO: Tactic failed: [critical] [/home/m8simmon/cs846/AI4EC/integration/output/experiments/run-20260807T151318Z/trials/trial_002/agent_work.agent.ec:829] an hypothesis or variable named `Adv_choose_ll` already exists
INFO: Trial 3: log_gen (3 tactic lines -> 10 step budget)
INFO: Trial 4: gen_log (5 tactic lines -> 12 step budget)
INFO: Trial 5: grexpAll (5 tactic lines -> 12 step budget)
INFO: Trial 6: RO_track_

KeyboardInterrupt: 

## Results

> ⚠️ **`successes` is not a repair rate.** With `elgamal-changelog-repair` it
> also counts trials that replayed verbatim with zero LLM calls. The table
> below separates the two, which is the distinction that matters.

In [ ]:
print(f"{'#':<3} {'Name':<20} {'Outcome':<10} {'Steps':>5} {'Calls':>6} {'Cost':>10}  Route")
print("-" * 78)
model_repairs = zero_llm = 0
for t in result.trial_results:
    calls = t.token_usage.calls
    cost = (t.estimated_cost or {}).get("usd", 0.0)
    if t.reason == "COMPLETE" and calls == 0:
        route, zero_llm = "replay (free)", zero_llm + 1
    elif t.reason == "COMPLETE":
        route, model_repairs = "MODEL REPAIR", model_repairs + 1
    else:
        route = "model, unsolved" if calls else "—"
    print(f"{t.trial_id:<3} {t.name:<20} {t.reason:<10} {t.steps:>5} {calls:>6} {cost:>10.6f}  {route}")

attempted = sum(1 for t in result.trial_results if t.token_usage.calls > 0)
print(f"\nZero-LLM replays        : {zero_llm}")
print(f"Repaired BY THE MODEL   : {model_repairs} of {attempted} attempted")

In [ ]:
import json

m = result.repair_metrics
if not m:
    print("No repair metrics (spec wrote no repair artifacts).")
else:
    if "replay" in m:
        r = m["replay"]
        print("How much of the 2020 proof still compiles")
        print(f"  fully replayed      : {r['fully_replayed']}/{r['trials']} ({r['fully_replayed_rate']:.0%})")
        print(f"  mean fraction kept  : {r['mean_replayed_fraction']:.1%}"
              f"  (min {r['min_replayed_fraction']:.1%}, max {r['max_replayed_fraction']:.1%})")
        print(f"  tactics accepted    : {r['total_tactics_accepted']}/{r['total_tactics']}")

    if "import_repair" in m:
        ir = m["import_repair"]
        # `resolved` (W4.5) is the headline: did the file get past LOADING,
        # which is the only thing import repair is responsible for. A file
        # whose one remaining complaint is a bad tactic has been handed to the
        # solver -- that is this module finishing, even though EasyCrypt still
        # exits nonzero. The old `improved` only asked whether the first error
        # moved later in the file, which almost any edit achieves.
        print(f"\nImport repair: {ir['resolved']}/{ir['attempted']} resolved "
              f"({ir['resolved_rate']:.0%})")
        for outcome, n in (ir.get("outcomes") or {}).items():
            meaning = {
                "loads": "compiles clean",
                "reached_proof": "load errors gone; a TACTIC is now at fault",
                "advanced": "still a load error, but a later/different one",
                "none": "nothing measurable changed",
                "regressed": "EasyCrypt stops earlier",
            }.get(outcome, "")
            print(f"    {outcome:<14} {n:>3}   {meaning}")
        remaining = ir.get("remaining_error_kinds") or {}
        if remaining:
            print(f"  still failing on   : {json.dumps(remaining)}")
            pre_proof = {k: v for k, v in remaining.items()
                         if k in {"unknown_theory", "unknown_symbol",
                                  "parse_error", "type_error", "unknown"}}
            print("  -> " + (
                f"{sum(pre_proof.values())} file(s) still have a PRE-PROOF error: "
                "the migration manifest has a gap here."
                if pre_proof else
                "no pre-proof errors left; every remaining failure is the solver's."
            ))
        print(f"  kept over original : {ir['worth_keeping']}/{ir['attempted']}"
              "   (a low bar -- it gates promotion, not success)")

    if "changelog_hops" in m:
        print(f"\nChangelog hops: {json.dumps(m['changelog_hops'])}")

    if "hint_uptake" in m:
        h = m["hint_uptake"]
        print("\nHint uptake")
        print(f"  used a hinted identifier : {h['trials_using_a_hinted_identifier']}/{h['trials_scored']}")
        print(f"  trials that landed a tactic (scorable): {h.get('trials_with_accepted_tactics')}")
        if not h.get("trials_with_accepted_tactics"):
            print("  -> NOT MEASURABLE: no trial accepted a tactic, so a 0 rate says")
            print("     nothing about whether the hints were useful.")
        print("  -> A PROXY either way: the model might have reached that name")
        print("     anyway. The counterfactual (CHANGELOG_HINTS = False) needs")
        print("     several seeds per arm to say anything, and an A/B on this")
        print("     corpus was already dropped for variance -- see")
        print("     docs/IMPLEMENTATION_PROGRESS.md section 11 item 6.")


## Tactics per lemma in the repaired files

In [ ]:
import re

_COMMENT = re.compile(r"\(\*.*?\*\)", re.DOTALL)


def tactics_in_proof(path):
    """Count tactic STATEMENTS in a repaired file.

    A statement closes at a `.` that is outside every bracket AND followed by
    whitespace. Both conditions earned their place:
      - `seq 5 5 : (invariant ...)` wraps across lines, so counting LINES is
        not counting tactics, and `wp; skip; smt().` is one line, three steps.
      - `RealOrder.lerr_eq` has a depth-0 dot with identifiers on both sides.
        Treating it as a terminator invented a "+1 by the model" on a lemma
        the agent never ran against.
    """
    if not path.is_file():
        return -1
    src = path.read_text(encoding="utf-8").splitlines()
    starts = [i for i, l in enumerate(src) if l.strip().startswith("proof")]
    if not starts:
        return -1
    body = []
    for l in src[max(starts) + 1:]:
        if l.strip().startswith("qed"):
            break
        body.append(l)
    text = _COMMENT.sub(" ", "\n".join(body))
    text = re.sub(r"\(\*(?:(?!\*\)).)*$", " ", text, flags=re.DOTALL)
    depth = n = 0
    seen = False
    for i, ch in enumerate(text):
        if ch in "([{":
            depth += 1
        elif ch in ")]}":
            depth = max(0, depth - 1)
        elif not ch.isspace():
            seen = True
        if ch == "." and depth == 0 and seen:
            nxt = text[i + 1] if i + 1 < len(text) else "\n"
            if nxt.isspace():
                n += 1
                seen = False
    return n


def _agent_steps(trial_dir):
    log = trial_dir / "agent_log.json"
    if not log.is_file():
        return 0
    return sum(1 for e in json.loads(log.read_text()).get("events", [])
               if e.get("event") == "iteration")


print(f"{'#':<3} {'lemma':<18} {'original':>8} {'replayed':>8} {'repaired':>8} {'model':>6}  outcome")
print("-" * 76)
_by_id = {t.trial_id: t for t in result.trial_results}
_total_o = _total_r = 0
_suspect = []
for _d in sorted((result.output_dir / "trials").iterdir()):
    if not _d.is_dir():
        continue
    _tid = int(_d.name.split("_")[1])
    _bp = _d / "bootstrap_result.json"
    _b = json.loads(_bp.read_text()) if _bp.is_file() else {}
    _o, _rep = _b.get("total_count", 0), _b.get("accepted_count", 0)
    _fix = tactics_in_proof(_d / "agent_work.agent.ec")
    _t = _by_id.get(_tid)
    _total_o += _o
    _total_r += max(_fix, 0)
    # GROUND TRUTH: nothing was added, so the count must equal the prefix.
    if _b.get("fully_replayed") and _agent_steps(_d) == 0 and _fix != _rep:
        _suspect.append((_d.name, _fix, _rep))
    print(f"{_tid:<3} {(_t.name if _t else '?'):<18} {_o:>8} {_rep:>8} {_fix:>8} "
          f"{_fix - _rep:>+6}  {_t.reason if _t else ''}")
print("-" * 76)
print(f"{'':<3} {'TOTAL':<18} {_total_o:>8} {'':>8} {_total_r:>8}")

if _suspect:
    print("\n*** COUNTER BUG ***  These trials replayed verbatim with zero agent")
    print("steps, so `repaired` MUST equal `replayed`. It does not:")
    for _n, _f, _r in _suspect:
        print(f"    {_n}: counted {_f}, replayed {_r}")
    print("Do not trust the `model` column until this is fixed.")
else:
    print("\ncalibration OK: every untouched trial counts exactly its replayed prefix.")
print("\n`repaired` < `replayed` means the run was cut off mid-repair (e.g. by")
print("the spend cap) after an undo -- that trial is not a usable data point.")

## Per-failure diagnostics

Which failures were proof-level vs import-level, and therefore which kind of
changelog evidence the retrieval routed to (`ec_errors.py` wiring).

In [ ]:
from collections import Counter

from integration.agent.ec_errors import classify_error

kinds, accepted_total, refreshes = Counter(), 0, 0
for trial_dir in sorted((result.output_dir / "trials").iterdir()):
    log = trial_dir / "agent_log.json"
    if not log.is_file():
        continue
    events = json.loads(log.read_text()).get("events", [])
    refreshes += sum(1 for e in events if e.get("event") == "changelog_hint_refresh")
    for e in events:
        if e.get("event") != "iteration" or e.get("action") != "tactic":
            continue
        if e.get("outcome") in ("accepted", "complete"):
            accepted_total += 1
        elif e.get("outcome") == "failed" and e.get("error"):
            kinds[classify_error(e["error"]).kind] += 1

print(f"Tactics accepted by the model : {accepted_total}")
print(f"Live changelog hint refreshes : {refreshes}")
print("\nFailure kinds (drives which evidence is retrieved):")
for kind, n in kinds.most_common():
    route = "tactic changelog entries" if kind in {"tactic_error", "proof_incomplete"} else "import evidence"
    print(f"  {kind:<18} {n:>3}   -> {route}")

## Inspect one trial

Set `TRIAL` to a trial that used the model.

In [ ]:
TRIAL = next((t.trial_id for t in result.trial_results if t.token_usage.calls), 0)
trial_dir = result.output_dir / "trials" / f"trial_{TRIAL:03d}"
print(f"=== {trial_dir.name} ===\n")

boot = trial_dir / "bootstrap_result.json"
if boot.is_file():
    b = json.loads(boot.read_text())
    print(f"Replayed {b['accepted_count']}/{b['total_count']} original tactics")
    if b.get("failed_tactic"):
        print(f"First tactic that no longer applies:\n  {b['failed_tactic'][:300]}\n")

# Where the §11 work is visible per trial: which error each rule was chosen
# against, and which were taken back out as unnecessary.
ir = trial_dir / "import_repair.json"
if ir.is_file():
    from integration.experiment.repair_metrics import _infer_outcome

    d = json.loads(ir.read_text())
    # Runs made before W4.5 have none of these keys. Derive what can be
    # derived and say so, rather than raising on your own older runs --
    # `_infer_outcome` deliberately never returns `reached_proof`, because
    # that needs a classification those artifacts do not carry.
    outcome = d.get("outcome") or _infer_outcome(d) + " (inferred: pre-W4.5 run)"
    before = f"{d.get('error_kind_before') or '?'}:{d.get('error_line_before', -1)}"
    after = f"{d.get('error_kind_after') or 'none'}:{d.get('error_line_after', -1)}"
    print(f"Import repair: {outcome}   ({before} -> {after})")
    for a in d.get("applied", []):
        mark = "keep" if a.get("kept") else "DROP"
        chosen = (f"chosen for {a['selected_for']}, relevance {a.get('relevance', 0)}"
                  if a.get("selected_for") else "")
        print(f"  [{mark}] {a.get('id', '?'):<44} {chosen}")
        print(f"         {a.get('reason', '')}")
    print()

hints = trial_dir / "changelog_hints.txt"
if hints.is_file():
    text = hints.read_text()
    print(f"--- hint block shown to the model ({len(text)} chars) ---")
    print(text[:1500])

log = trial_dir / "agent_log.json"
if log.is_file():
    print("\n--- what the model tried ---")
    for e in json.loads(log.read_text()).get("events", []):
        if e.get("event") == "iteration" and e.get("action") == "tactic":
            print(f"  {e.get('outcome'):<9} {(e.get('tactic') or '')[:70]}")

## Comparing providers

Re-run with `PROVIDER = "anthropic"` / `MODEL = "claude-opus-5"` and compare.
`sort_by_difficulty=True` makes the case order deterministic, so runs are
directly comparable without a shared seed.

The three numbers worth comparing:

1. **Repaired by the model** — not `successes`, which includes free replays.
2. **`hint_uptake.rate_among_scorable`** — only meaningful once a run actually
   lands tactics; `None` means not measurable.
3. **`estimated_cost.usd` per repair** — a model that repairs twice as much for
   ten times the cost is a different trade, not a better one.

### Comparing two runs: mostly, don't

Two runs of this notebook cannot be compared. Run-to-run spread under
**identical** configuration reached 11-vs-1 accepted tactics on this corpus
(`docs/ELGAMAL_E2E_RESULTS.md` §6), which is larger than any difference a pair
of runs could show — that is why the `show_remaining_original` A/B was dropped
rather than deferred.

If you have several seeds per arm anyway, score them with the tool rather than
by eye, because it encodes exactly that limit:

```bash
python3 -m integration.experiment.compare_runs \
  --arm a runs/a-* --arm b runs/b-*
```

It reports `CONCLUSIVE` only when the gap between arm means exceeds the widest
within-arm range, never with one run per arm, and warns about mismatched seeds,
mixed models, and runs stopped early by their spend cap.

### Version hopping is CLI-only, on purpose

`--version-hop` localizes which release broke a tactic, but provisioning a
release is an opam switch plus a full OCaml build -- minutes and hundreds of MB
each. That is not something to put behind a notebook cell. See
`docs/IMPLEMENTATION_PROGRESS.md` §9.2.


In [ ]:
summary_path = result.output_dir / "summary.json"
print(f"Full summary: {summary_path}")
print(json.dumps({
    "spec": result.spec_name,
    "provider": agent.llm_provider,
    "model": agent.llm_model,
    "thinking": agent.llm_thinking,
    "effort": agent.llm_reasoning_effort,
    "repaired_by_model": model_repairs,
    "attempted_by_model": attempted,
    "zero_llm_replays": zero_llm,
    "cost_usd": (result.estimated_cost or {}).get("usd"),
    "hint_uptake": result.repair_metrics.get("hint_uptake"),
    # Which arm this run was, so a later compare_runs.py can pair it.
    "arm": result.arm,
}, indent=2))